# Frequency construction laboratory

This notebook studies the first transformation performed by MFDRO: converting one panel of daily simple returns into several empirical frequency measures. The objective is to make aggregation conventions, effective horizons, sample sizes, and their consequences observable before any transport calculation.

All data are synthetic, all seeds are fixed, and the notebook requires no network access.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from mfdro import (
    FrequencySpec,
    MultiFrequencySignal,
    SignalConfig,
    build_frequency_measures,
    compound_returns,
)

plt.style.use("seaborn-v0_8-whitegrid")
COLORS = ["#183b4e", "#008c82", "#d17a22", "#6f5b9a", "#9b3a4a"]

## 1. A controlled daily panel

The simulation combines a persistent common factor and asset-specific noise. It is rich enough to produce multivariate dependence while retaining a fully known data-generating process. Values are daily simple returns.

In [ ]:
rng = np.random.default_rng(20250301)
dates = pd.bdate_range("2019-01-01", "2024-12-31")
time = np.arange(len(dates), dtype=float)
volatility = 0.006 + 0.002 * (1.0 + np.sin(time / 90.0))
factor = rng.normal(size=len(dates)) * volatility
loadings = np.linspace(0.65, 1.20, 7)
noise = rng.normal(0.0, 0.005, size=(len(dates), len(loadings)))
returns = pd.DataFrame(
    0.00015 + factor[:, None] * loadings + noise,
    index=dates,
    columns=[f"asset_{index:02d}" for index in range(len(loadings))],
)

assert returns.index.is_monotonic_increasing
assert not returns.isna().any().any()
returns.describe().T

In [ ]:
wealth = (1.0 + returns.iloc[:, :4]).cumprod()
axis = wealth.plot(figsize=(10, 3.5), color=COLORS[:4], linewidth=1.1)
axis.set(title="Synthetic cumulative wealth indices", xlabel="date", ylabel="growth of 1")
plt.tight_layout()

## 2. Declare five frequencies

`rule` controls calendar grouping; `horizon` controls subsequent power scaling. They are deliberately separate scientific choices. The first specification is always the unaggregated base panel.

In [ ]:
FIVE_FREQUENCIES = (
    FrequencySpec("daily", 1.0),
    FrequencySpec("weekly", 5.0, rule="W-FRI", closed="right", label="right"),
    FrequencySpec("biweekly", 10.0, rule="2W-FRI", closed="right", label="right"),
    FrequencySpec("monthly", 21.0, rule="ME", closed="right", label="right"),
    FrequencySpec("quarterly", 63.0, rule="QE", closed="right", label="right"),
)

measures = build_frequency_measures(returns, frequency_specs=FIVE_FREQUENCIES)
summary = pd.DataFrame(
    {
        "rule": [spec.rule or "unaggregated" for spec in FIVE_FREQUENCIES],
        "effective_horizon": [spec.horizon for spec in FIVE_FREQUENCIES],
        "observations": [len(measures[spec.name]) for spec in FIVE_FREQUENCIES],
        "mean_asset_std": [measures[spec.name].std().mean() for spec in FIVE_FREQUENCIES],
    },
    index=[spec.name for spec in FIVE_FREQUENCIES],
)
summary

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(11, 3.8))
summary["observations"].plot.bar(ax=axes[0], color=COLORS, rot=25)
axes[0].set(title="Empirical sample size", xlabel="frequency", ylabel="observations")
axes[1].boxplot(
    [frame.to_numpy().ravel() for frame in measures.values()],
    showfliers=False,
)
axes[1].set_xticks(range(1, len(measures) + 1), labels=list(measures))
axes[1].tick_params(axis="x", rotation=25)
axes[1].set(title="Return distributions before scaling", xlabel="frequency", ylabel="simple return")
figure.tight_layout()

## 3. Verify the aggregation formula

Simple returns compound multiplicatively. Summing them would be a different approximation and is therefore not offered as an interchangeable aggregation mode.

In [ ]:
tiny = pd.DataFrame(
    {"asset": [0.10, -0.05, 0.02]},
    index=pd.to_datetime(["2024-01-02", "2024-01-03", "2024-01-04"]),
)
observed = float(compound_returns(tiny, "ME").iloc[0, 0])
expected = 1.10 * 0.95 * 1.02 - 1.0

assert np.isclose(observed, expected)
pd.Series({"compounded": observed, "simple_sum": float(tiny["asset"].sum())})

## 4. Two, three, and five-frequency signals

The following comparison demonstrates configurability, not a model-selection rule. Changing the grid changes the scientific object being measured. All variants use the same projected seed.

In [ ]:
FREQUENCY_GRIDS = {
    "2: daily/monthly": (FIVE_FREQUENCIES[0], FIVE_FREQUENCIES[3]),
    "3: daily/weekly/monthly": (FIVE_FREQUENCIES[0], FIVE_FREQUENCIES[1], FIVE_FREQUENCIES[3]),
    "5: full grid": FIVE_FREQUENCIES,
}
records = []
for label, specs in FREQUENCY_GRIDS.items():
    config = SignalConfig.projected(
        frequency_specs=specs,
        n_projections=96,
        n_quantiles=96,
        random_state=20250301,
    )
    estimate = MultiFrequencySignal(config).estimate(
        build_frequency_measures(returns, frequency_specs=specs),
        seed=314159,
    )
    records.append(
        {
            "grid": label,
            "rho": estimate.rho,
            "sqrt_rho": estimate.sqrt_rho,
            "digest": config.digest[:12],
        }
    )

grid_results = pd.DataFrame(records).set_index("grid")
grid_results

In [ ]:
axis = grid_results["sqrt_rho"].plot.bar(figsize=(8, 3.5), color=COLORS[:3], rot=15)
axis.set(
    title="Configured frequency grid changes the measured dispersion",
    xlabel="",
    ylabel=r"$\sqrt{\rho}$",
)
plt.tight_layout()

## 5. Boundary conventions are part of identity

A change from right-closed to left-closed weekly bins can change which daily observation enters each period. MFDRO therefore includes this convention in the configuration digest.

In [ ]:
right_week = FrequencySpec("weekly", 5.0, rule="W-FRI", closed="right", label="right")
left_week = FrequencySpec("weekly", 5.0, rule="W-FRI", closed="left", label="right")
right_config = SignalConfig.projected(frequency_specs=(FIVE_FREQUENCIES[0], right_week))
left_config = SignalConfig.projected(frequency_specs=(FIVE_FREQUENCIES[0], left_week))
restored_week = FrequencySpec.from_dict(right_week.to_dict())

assert restored_week == right_week
assert right_config.digest != left_config.digest
pd.DataFrame(
    {
        "closed": [right_week.closed, left_week.closed],
        "config_digest": [right_config.digest, left_config.digest],
    },
    index=["right convention", "left convention"],
)

## Interpretation

Longer aggregation periods provide fewer, larger-magnitude observations. Horizon scaling later places them on the configured comparable scale; it does not recreate the information lost through aggregation. Frequency grids and boundary conventions should therefore be declared from the research design and retained in configuration JSON, not selected from whichever path produces the most attractive downstream result.